# CredituS — Algoritmo Genético para Classificação de Crédito

Perceptron treinado por **algoritmo genético** para classificar clientes como adimplentes (1) ou inadimplentes (0), seguindo a metodologia Prandiano.

**Ideia central:** cada cromossomo é um vetor $[b, w_1, \dots, w_n]$ que define um perceptron. A hipótese é
$$q = \mathbf{w} \cdot \mathbf{x} + b, \qquad \hat{y} = \begin{cases}1 & \text{se } q \geq 0 \\ 0 & \text{caso contrário}\end{cases}$$

A **função fitness** é o produto das taxas de acerto por classe:
$$f = PA \times PI$$
Esse produto força o modelo a acertar **as duas classes simultaneamente** — se ele "chuta tudo 1" para maximizar $PA$, o $PI$ vai a zero e o fitness zera junto.


## 1. Imports e carregamento da base

Convenção da planilha `basedados.xlsx`:
- 1ª coluna → índice do cliente (descartada)
- Colunas do meio → *features*
- Última coluna → gabarito (0/1)

`qtd_genes = qtd_features + 1` porque cada cromossomo carrega **um gene extra para o bias**.


In [ ]:
import numpy as np
import pandas as pd

df  = pd.read_excel("basedados.xlsx")

dataframe_dados_clientes = df.iloc[:, 1:-1]  # ignora primeira coluna (índice) e última (gabarito)
dataframe_gabarito       = df.iloc[:, -1]    # sempre a última coluna

array_dados_clientes = dataframe_dados_clientes.values
array_gabarito       = dataframe_gabarito.values

qtd_features = array_dados_clientes.shape[1]
qtd_genes    = qtd_features + 1  # features + bias

print(f"Base carregada: {array_dados_clientes.shape[0]} clientes, {qtd_features} features")

## 2. População inicial — `criar_cromossomos`

Gera uma matriz `(qtd_cromossomos × qtd_genes)` com valores uniformes em $[-1, 1]$.

A transformação `-1 + 2 * rand()` mapeia $[0, 1) \to [-1, 1)$ — pesos iniciais centrados no zero, sem viés de sinal.


In [ ]:
def criar_cromossomos(qtd_cromossomos: int=6, qtd_genes: int=19) -> np.ndarray:
    return -1 + 2 * np.random.rand(qtd_cromossomos, qtd_genes)

## 3. Avaliação — `calcular_fitness`

Para cada cromossomo da população:

1. Separa **bias** (primeiro gene) e **pesos** (restante).
2. Calcula $\mathbf{q} = X\mathbf{w} + b$ para todos os clientes de uma vez (produto matriz-vetor vetorizado).
3. Aplica o degrau com `np.where`: $\hat{y}_i = 1$ se $q_i \geq 0$, senão 0.
4. Conta acertos por classe usando máscaras booleanas (`&` é AND elemento-a-elemento).
5. Calcula $PA$, $PI$ e o fitness $f = PA \cdot PI$.

Retorna um vetor com um fitness por cromossomo.


In [ ]:
def calcular_fitness(cromossomos: np.ndarray, array_dados_clientes: np.ndarray, array_gabarito: np.ndarray) -> np.ndarray:
    """
    Pega cada linha do array_cromossomos (menos o primeiro termo, que é o bias) e itera sobre cada linha da array de dados clientes
    cada linha (iteracao) e um produto escalar
    cada iteracao vai gerar um vetor coluna contendo 0 e 1 chamado de vetor hipotese

    depois comparar o vetor hipotese de cada cromossomo com o vetor gabarito e calcular a porcentagem de acerto

    depois calcular o fitness seguindo a formula 

    percentual_adimplente = quantidade de 1 no vetor hipotese/ total de 1 no gabarito
    percentual_inadimplente - quantidade de 0 no vetor hipotese/ total de 0 no gabarito

    fitness = percentual_adimplente * percentual_inadimplente

    cada cromossomo vai ter um fitness. Somar todos os fitness e calcular a porcentagem relativa de acerta de cada cromossomo
    com base nisso
    """

    total_adimplentes = np.sum(array_gabarito == 1)
    total_inadimplentes = np.sum(array_gabarito == 0)
    
    lista_hipotese = []

    #iteracao pra cada cromossomo
    for linha in cromossomos:  # o for num array vai de linha em linha automaticamente
        
        bias = linha[0]
        genes = linha[1:]

        q = np.dot(array_dados_clientes, genes) + bias

        #claudio ajudou, onde cada elemento de Q for maior igual a zero troque por 1, se nao troque por zero
        vetor_hipotese = np.where(q >= 0, 1, 0)

        acertos_adimplentes = np.sum((vetor_hipotese == 1) & (array_gabarito == 1)) #se a hipotese for 1 e o gabarito for 1 ele acertou, entao contabiliza
        acertos_inadimplentes = np.sum((vetor_hipotese == 0) & (array_gabarito == 0)) #se a hipotese for 0 e o gabarito for 0 ele acertou, entao contabiliza
        # se nao o & da false e ele nao soma 


        percentual_adimplente = acertos_adimplentes / total_adimplentes
        percentual_inadimplente = acertos_inadimplentes / total_inadimplentes

        fitness = percentual_adimplente * percentual_inadimplente
        
        lista_hipotese.append(fitness)
        vetor_fitness = np.array(lista_hipotese) 

    return vetor_fitness # cada cromossomo tem um fitness, entao o vetor de fitness tem o mesmo numero de linhas do array de cromossomos

## 4. Normalização para a roleta — `fitness_percentual`

Converte o vetor de fitnesses em **probabilidades** (soma = 1) dividindo cada valor pela soma total.

Se todos os fitnesses forem zero (pode acontecer na geração 0), distribui probabilidade **uniforme** — assim `cumsum` não gera NaN e a roleta continua funcionando.


In [ ]:
def fitness_percentual(vetor_fitnesses: np.ndarray) -> np.ndarray:
    soma = np.sum(vetor_fitnesses)
    if soma == 0:
        return np.ones(len(vetor_fitnesses)) / len(vetor_fitnesses)
    return vetor_fitnesses / soma

## 5. Seleção por roleta — `selecionar_pais_roleta`

**Analogia da roleta:** imagine uma roleta onde cada cromossomo ocupa uma fatia proporcional ao seu fitness. Cromossomos bons ocupam fatias grandes → probabilidade maior de serem sorteados.

**Implementação:**
- `cumsum(percentuais)` = fronteiras das fatias no intervalo $[0, 1]$.
- `np.random.rand()` sorteia um ponto na roleta.
- `searchsorted` encontra em qual fatia esse ponto caiu — **O(log n)** via busca binária.

O `min(..., len-1)` é uma proteção numérica: se `rand()` retornar algo muito próximo de 1 e a `cumsum` fechar em 0.99999..., o `searchsorted` pode devolver um índice fora do array.


In [ ]:
def selecionar_pais_roleta(
    cromossomos: np.ndarray,
    percentual_fitnesses: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    roleta_acumulada = np.cumsum(percentual_fitnesses)

    indice_pai = min(int(np.searchsorted(roleta_acumulada, np.random.rand())), len(cromossomos) - 1)
    indice_mae = min(int(np.searchsorted(roleta_acumulada, np.random.rand())), len(cromossomos) - 1)

    return cromossomos[indice_pai], cromossomos[indice_mae]

## 6. Cruzamento — `cruzar_pais`

**Crossover de 1 ponto**, executado 3 vezes com pontos de corte independentes:

$$\text{filho}_k = [\underbrace{p_1, \dots, p_{c_k}}_{\text{do pai}}, \underbrace{m_{c_k+1}, \dots, m_n}_{\text{da mãe}}]$$

Gerar 3 filhos por casal aumenta a diversidade a cada geração — depois só os 2 melhores entram na população (ver célula 8).

O `randint(1, len(pai))` começa em 1 (não em 0) pra garantir que sempre haja **algum** gene herdado do pai.


In [ ]:
def cruzar_pais(pai: np.ndarray, mae: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    c1 = np.random.randint(1, len(pai))
    c2 = np.random.randint(1, len(pai))
    c3 = np.random.randint(1, len(pai))

    filho1 = np.concatenate([pai[:c1], mae[c1:]])
    filho2 = np.concatenate([pai[:c2], mae[c2:]])
    filho3 = np.concatenate([pai[:c3], mae[c3:]])

    return filho1, filho2, filho3

## 7. Mutação — `mutar`

Para cada filho, sorteia **um gene aleatório** e substitui por um novo valor uniforme em $[-1, 1]$.

Taxa de mutação implícita = $1/n$ por indivíduo — é o mecanismo que impede convergência prematura em um ótimo local, injetando aleatoriedade a cada geração.


In [ ]:
def mutar(filho1: np.ndarray, filho2: np.ndarray, filho3: np.ndarray,) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    for filho in [filho1, filho2, filho3]:
        indice = np.random.randint(0, len(filho))
        filho[indice] = -1 + 2 * np.random.rand()

    return filho1, filho2, filho3

## 8. Atualização — `atualizar_populacao`

Estratégia **steady-state com elitismo por substituição**:

1. Avalia os 3 filhos gerados.
2. `argsort(...)[-2:]` → índices dos **2 melhores filhos**.
3. `argsort(...)[:2]` → índices dos **2 piores da população atual**.
4. Substitui os 2 piores pelos 2 melhores filhos.

Efeito: os bons cromossomos **nunca são perdidos** (elitismo implícito), e a pressão seletiva é forte porque só os melhores dos filhos entram.


In [ ]:
def atualizar_populacao(cromossomos: np.ndarray,vetor_fitnesses: np.ndarray,filho1: np.ndarray,filho2: np.ndarray,filho3: np.ndarray,array_dados_clientes: np.ndarray,array_gabarito: np.ndarray,) -> np.ndarray:
    filhos = np.array([filho1, filho2, filho3])
    fitnesses_filhos = calcular_fitness(filhos, array_dados_clientes, array_gabarito)

    indices_melhores_filhos = np.argsort(fitnesses_filhos)[-2:] # pega os indices dos 2 melhores filhos, ou seja, os 2 filhos com maior fitness
    indices_piores          = np.argsort(vetor_fitnesses)[:2] # pega os indices dos 2 piores cromossomos da populacao, ou seja, os 2 cromossomos com menor fitness

    nova_populacao = cromossomos.copy()
    for i in range(2):
        idx_pior  = int(indices_piores[i])
        idx_filho = int(indices_melhores_filhos[i])
        nova_populacao[idx_pior] = filhos[idx_filho]

    return nova_populacao

## 9. Loop principal — `algoritmo_genetico`

Junta tudo. A cada geração:

1. Avalia a população (`calcular_fitness`).
2. Normaliza para probabilidades (`fitness_percentual`).
3. Guarda o **melhor global** encontrado até agora (elitismo externo — protege contra o azar de o melhor ser destruído).
4. Testa critério de parada (`fitness_alvo`).
5. Seleciona pais → cruza → muta → atualiza população.

**Dois critérios de parada:** atingiu `fitness_alvo` **ou** esgotou `geracoes`.


In [ ]:
def algoritmo_genetico(array_dados_clientes: np.ndarray, array_gabarito: np.ndarray, qtd_cromossomos: int = 6, qtd_genes: int= 19, geracoes: int= 100, fitness_alvo: float = 0.90,) -> tuple[np.ndarray, float]:

    populacao = criar_cromossomos(qtd_cromossomos, qtd_genes)

    melhor_cromossomo = None
    melhor_fitness    = 0.0

    for geracao in range(geracoes):

        fitnesses   = calcular_fitness(populacao, array_dados_clientes, array_gabarito)
        percentuais = fitness_percentual(fitnesses)

        idx_melhor    = int(np.argmax(fitnesses))
        fitness_atual = float(fitnesses[idx_melhor])

        if fitness_atual > melhor_fitness:
            melhor_fitness    = fitness_atual
            melhor_cromossomo = populacao[idx_melhor].copy()

        print(f"Geração {geracao+1:>3} | melhor fitness: {melhor_fitness:.4f}")

        if melhor_fitness >= fitness_alvo:
            print(f"\nFitness alvo {fitness_alvo} atingido na geração {geracao+1}.")
            break

        pai, mae = selecionar_pais_roleta(populacao, percentuais)
        filho1, filho2, filho3 = cruzar_pais(pai, mae)
        filho1, filho2, filho3 = mutar(filho1, filho2, filho3)
        populacao = atualizar_populacao(
            populacao, fitnesses,
            filho1, filho2, filho3,
            array_dados_clientes, array_gabarito,
        )

    return melhor_cromossomo, melhor_fitness

## 10. Inferência — `prever_novo_cliente`

Depois de treinado, o melhor cromossomo **é o modelo**. Para classificar um cliente novo basta aplicar a mesma regra do perceptron:

$$\hat{y} = \mathbf{1}\{ \mathbf{w} \cdot \mathbf{x}_{\text{novo}} + b \geq 0 \}$$

Repare que não tem `for`: é uma única avaliação. Todo o custo computacional ficou no treino; a inferência é praticamente de graça.


In [ ]:
def prever_novo_cliente(melhor_cromossomo: np.ndarray, dados_novo_cliente: np.ndarray) -> int:
    """
    Usa o melhor cromossomo encontrado pelo algoritmo genético para prever
    se um novo cliente será adimplente (1) ou inadimplente (0).
    
    Args:
        melhor_cromossomo: Array 1D retornado pelo algoritmo_genetico.
        dados_novo_cliente: Array 1D com as features do cliente (sem o gabarito).
        
    Returns:
        1 (Adimplente) ou 0 (Inadimplente)
    """
    # Separa o bias dos pesos (genes)
    bias = melhor_cromossomo[0]
    genes = melhor_cromossomo[1:]
    
    # Calcula o produto escalar (Q)
    q = np.dot(dados_novo_cliente, genes) + bias
    
    # Aplica a regra de decisão
    previsao = 1 if q >= 0 else 0
    
    return previsao

## 11. Execução do treino

Roda o algoritmo com até **1000 gerações** e critério de parada em fitness $\geq 0.95$.

`resultado` é uma tupla `(melhor_cromossomo, melhor_fitness)` — desempacote com `melhor_cromossomo, melhor_fitness = resultado` para usar depois em `prever_novo_cliente`.


In [ ]:
resultado = algoritmo_genetico(array_dados_clientes, array_gabarito, geracoes=1000, fitness_alvo=0.95)

## 12. Espaço para testes

Célula livre para experimentar previsões em novos clientes, análise dos pesos aprendidos, etc.
